# Pertussis Sanity Check
https://www.cdc.gov/pertussis/php/surveillance/index.html?CDC_AA_refVal=https%3A%2F%2Fwww.cdc.gov%2Fpertussis%2Fphp%2Fsurveillance%2Fpertussis-incidence-by-age-group-and-year.html

In [163]:
import pandas as pd

pertussis_df = pd.read_csv('../app/data/pertussis_age.csv')
ms_df = pd.read_csv('../app/data/cases_coverage_ms.csv')

In [164]:
ms_df.head()

,year,state,measles_cases,mumps_cases,pertussis_cases,measles_coverage_pct,measles_n,measles_n_vaccinated,mumps_coverage_pct,mumps_n,mumps_n_vaccinated,pertussis_coverage_pct,pertussis_n,pertussis_n_vaccinated,population,measles_cases_per_100k,mumps_cases_per_100k,pertussis_cases_per_100k
0,1995,AK,NaN,13.0,1.0,89.86,194.0,177.0,89.86,194.0,177.0,77.58,194.0,159.0,604412.0,NaN,2.150851,0.165450
1,1996,AK,63.0,3.0,4.0,84.55,260.0,225.0,84.55,260.0,225.0,78.25,260.0,210.0,608569.0,10.352154,0.492960,0.657280
2,1997,AK,48.0,4.0,14.0,87.41,291.0,257.0,87.41,291.0,257.0,80.98,291.0,242.0,612968.0,7.830751,0.652563,2.283969
3,1998,AK,29.0,2.0,15.0,87.06,34.0,30.0,87.06,34.0,30.0,82.01,34.0,28.0,619932.0,4.677932,0.322616,2.419620
4,1999,AK,29.0,3.0,5.0,90.67,349.0,321.0,90.67,349.0,321.0,83.54,349.0,296.0,624779.0,4.641641,0.480170,0.800283


In [165]:
vacc_coverage_df = ms_df[['year', 'pertussis_coverage_pct']]
avg_vacc_coverage_df = vacc_coverage_df.groupby('year').mean()
len(avg_vacc_coverage_df)

23

In [166]:
vacc_coverage_df = ms_df[['year', 'pertussis_coverage_pct']]
min_vacc_coverage_df = vacc_coverage_df.groupby('year').min()
len(min_vacc_coverage_df)

23

In [167]:
vacc_coverage_df = ms_df[['year', 'pertussis_coverage_pct']]
max_vacc_coverage_df = vacc_coverage_df.groupby('year').max()
len(max_vacc_coverage_df)

23

In [168]:
pertussis_df.head()

,Year,<1 yr,1-6 yrs,7-10 yrs,11-19 yrs,20+ yrs,Overall Incidence,Data Status
0,1990,47.89,5.57,1.68,1.38,0.19,1.83,Finalized
1,1991,27.27,3.20,0.96,0.58,0.14,1.07,Finalized
2,1992,38.41,3.93,1.95,1.99,0.27,1.59,Finalized
3,1993,68.00,6.59,3.26,2.08,0.33,2.53,Finalized
4,1994,42.61,4.35,2.26,2.30,0.27,1.75,Finalized


In [169]:
child_pertussis_cases_df = pertussis_df[['Year', '<1 yr', '1-6 yrs']]
child_pertussis_cases_df.rename(columns={'Year': 'year'}, inplace=True)
child_pertussis_cases_df.head()

,year,<1 yr,1-6 yrs
0,1990,47.89,5.57
1,1991,27.27,3.20
2,1992,38.41,3.93
3,1993,68.00,6.59
4,1994,42.61,4.35


In [170]:
sanity_df = (
    avg_vacc_coverage_df.rename(
        columns=lambda col: f'avg_{col}' if col != 'year' else col
    )
    .merge(
        max_vacc_coverage_df.rename(
            columns=lambda col: f'max_{col}' if col != 'year' else col
        ),
        on='year',
        how='outer'
    )
    .merge(
        min_vacc_coverage_df.rename(
            columns=lambda col: f'min_{col}' if col != 'year' else col
        ),
        on='year',
        how='outer'
    )
    .merge(
        child_pertussis_cases_df,
        on='year',
        how='outer'
    )
    .sort_values('year')
    .reset_index(drop=True)
)

In [171]:
sanity_df = sanity_df[
    sanity_df["year"].between(1995, 2017)
].reset_index(drop=True)

In [172]:
sanity_df.head()

,year,avg_pertussis_coverage_pct,max_pertussis_coverage_pct,min_pertussis_coverage_pct,<1 yr,1-6 yrs
0,1995,79.347551,91.29,68.32,48.74,4.71
1,1996,81.532600,92.14,70.03,63.23,6.01
2,1997,82.680000,93.71,73.66,52.73,4.49
3,1998,84.052000,95.88,68.61,56.71,5.45
4,1999,84.413400,94.81,74.97,57.12,4.57


In [173]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def graph_pertussis_age_coverage(df):

    df = df.sort_values("year")

    fig = make_subplots(
        specs=[[{"secondary_y": True}]]
    )

    colors = {
        "<1 yr": "#E74C3C",
        "1-6 yrs": "#F39C12",
    }

    # Three grouped incidence bars per year
    for age_group, color in colors.items():

        fig.add_trace(
            go.Bar(
                x=df["year"],
                y=df[age_group],
                name=age_group,
                marker_color=color,
                hovertemplate=(
                    "Year: %{x}<br>"
                    f"Age group: {age_group}<br>"
                    "Incidence: %{y:.2f}"
                    "<extra></extra>"
                )
            ),
            secondary_y=False
        )

    # Three vaccination coverage lines
    coverage_lines = {
        "avg_pertussis_coverage_pct": {
            "name": "Average DTaP Coverage (%)",
            "color": "#3498DB",
            "dash": "solid"
        },
        "max_pertussis_coverage_pct": {
            "name": "Maximum DTaP Coverage (%)",
            "color": "#27AE60",
            "dash": "dash"
        },
        "min_pertussis_coverage_pct": {
            "name": "Minimum DTaP Coverage (%)",
            "color": "#E67E22",
            "dash": "dot"
        }
    }

    for column, settings in coverage_lines.items():

        fig.add_trace(
            go.Scatter(
                x=df["year"],
                y=df[column],
                name=settings["name"],
                mode="lines+markers",
                line=dict(
                    color=settings["color"],
                    width=3,
                    dash=settings["dash"]
                ),
                marker=dict(size=7),
                hovertemplate=(
                    "Year: %{x}<br>"
                    f"{settings['name']}: "
                    "%{y:.2f}%"
                    "<extra></extra>"
                )
            ),
            secondary_y=True
        )

    fig.update_yaxes(
        title_text="Pertussis Incidence (Cases per 100K)",
        secondary_y=False
    )

    fig.update_yaxes(
        title_text="Vaccination Coverage (%)",
        range=[0, 100],
        secondary_y=True
    )

    fig.update_xaxes(
        title_text="Year",
        dtick=1
    )

    fig.update_layout(
        title="National Pertussis Incidence by Age Group & DTaP Coverage",
        barmode="group",
        template="plotly_white",
        height=550,
        hovermode="x unified",
        legend=dict(
            orientation="h",
            y=1.12,
            x=0
        ),
        margin=dict(t=110)
    )

    return fig

In [174]:
fig = graph_pertussis_age_coverage(sanity_df)
fig.show()

In [175]:
import pandas as pd
import numpy as np
from scipy.stats import pearsonr

def pearson_pertussis_age(
    df,
    start_year=None,
    end_year=None,
    lag_years=0
):

    age_groups = ["<1 yr", "1-6 yrs"]

    coverage_columns = {
        "avg_pertussis_coverage_pct": "Average",
        "max_pertussis_coverage_pct": "Maximum",
        "min_pertussis_coverage_pct": "Minimum"
    }

    if not isinstance(lag_years, int) or lag_years < 0:
        raise ValueError("lag_years must be a non-negative integer")

    results = []

    data = df.copy().sort_values("year")

    # Filter incidence years
    if start_year is not None:
        data = data[data["year"] >= start_year]

    if end_year is not None:
        data = data[data["year"] <= end_year]

    # Keep coverage from original dataset
    coverage_data = df[["year"] + list(coverage_columns)].copy()

    # Shift coverage years forward to match future incidence
    coverage_data["year"] = coverage_data["year"] + lag_years

    # Match coverage from year t-lag with incidence in year t
    data = data[["year"] + age_groups].merge(
        coverage_data,
        on="year",
        how="left",
        validate="one_to_one"
    )

    # Six analyses: 2 age groups x 3 coverage measures
    for age_group in age_groups:

        for coverage_col, coverage_name in coverage_columns.items():

            valid = data[
                ["year", coverage_col, age_group]
            ].dropna()

            x = valid[coverage_col]
            y = valid[age_group]

            n = len(valid)

            if n >= 3 and x.nunique() > 1 and y.nunique() > 1:
                r, p = pearsonr(x, y)
            else:
                r, p = np.nan, np.nan

            results.append({
                "age_group": age_group,
                "coverage_type": coverage_name,
                "lag_years": lag_years,
                "start_year": start_year,
                "end_year": end_year,
                "n": n,
                "pearson_r": r,
                "p_value": p,
                "significant_0_05": (
                    bool(p < 0.05) if np.isfinite(p) else False
                )
            })

    return pd.DataFrame(results)

In [176]:
pearson_results = pearson_pertussis_age(
    sanity_df,
    start_year=1995,
    end_year=2017
)

display(pearson_results)

,age_group,coverage_type,lag_years,start_year,end_year,n,pearson_r,p_value,significant_0_05
0,<1 yr,Average,0,1995,2017,23,0.269977,0.212819,False
1,<1 yr,Maximum,0,1995,2017,23,-0.025200,0.909131,False
2,<1 yr,Minimum,0,1995,2017,23,0.349298,0.102324,False
3,1-6 yrs,Average,0,1995,2017,23,0.297041,0.168693,False
4,1-6 yrs,Maximum,0,1995,2017,23,-0.160653,0.464001,False
5,1-6 yrs,Minimum,0,1995,2017,23,0.518546,0.011244,True


In [177]:
pearson_pertussis_age(sanity_df, 1995, 2017, lag_years=1)

,age_group,coverage_type,lag_years,start_year,end_year,n,pearson_r,p_value,significant_0_05
0,<1 yr,Average,1,1995,2017,22,0.195379,0.383559,False
1,<1 yr,Maximum,1,1995,2017,22,-0.338482,0.123355,False
2,<1 yr,Minimum,1,1995,2017,22,0.280470,0.206125,False
3,1-6 yrs,Average,1,1995,2017,22,0.308896,0.161884,False
4,1-6 yrs,Maximum,1,1995,2017,22,-0.360769,0.099051,False
5,1-6 yrs,Minimum,1,1995,2017,22,0.476279,0.025039,True


In [181]:
pearson_pertussis_age(sanity_df, 1995, 2017, lag_years=2)

,age_group,coverage_type,lag_years,start_year,end_year,n,pearson_r,p_value,significant_0_05
0,<1 yr,Average,2,1995,2017,21,0.110191,0.634437,False
1,<1 yr,Maximum,2,1995,2017,21,-0.401315,0.071370,False
2,<1 yr,Minimum,2,1995,2017,21,0.316435,0.162260,False
3,1-6 yrs,Average,2,1995,2017,21,0.230571,0.314636,False
4,1-6 yrs,Maximum,2,1995,2017,21,-0.451334,0.040005,True
5,1-6 yrs,Minimum,2,1995,2017,21,0.473422,0.030178,True


In [179]:
pearson_pertussis_age(sanity_df, 1995, 2017, lag_years=3)

,age_group,coverage_type,lag_years,start_year,end_year,n,pearson_r,p_value,significant_0_05
0,<1 yr,Average,3,1995,2017,20,-0.022614,0.924607,False
1,<1 yr,Maximum,3,1995,2017,20,-0.453006,0.044876,True
2,<1 yr,Minimum,3,1995,2017,20,0.101805,0.669319,False
3,1-6 yrs,Average,3,1995,2017,20,0.170432,0.472509,False
4,1-6 yrs,Maximum,3,1995,2017,20,-0.421291,0.064321,False
5,1-6 yrs,Minimum,3,1995,2017,20,0.298773,0.200690,False


In [180]:
pearson_pertussis_age(sanity_df, 1995, 2017, lag_years=4)

,age_group,coverage_type,lag_years,start_year,end_year,n,pearson_r,p_value,significant_0_05
0,<1 yr,Average,4,1995,2017,19,0.137296,0.575141,False
1,<1 yr,Maximum,4,1995,2017,19,-0.460472,0.047259,True
2,<1 yr,Minimum,4,1995,2017,19,0.326028,0.173130,False
3,1-6 yrs,Average,4,1995,2017,19,0.271734,0.260424,False
4,1-6 yrs,Maximum,4,1995,2017,19,-0.498167,0.029951,True
5,1-6 yrs,Minimum,4,1995,2017,19,0.457481,0.048904,True
